# Lezione 11 — Drift e riaddestramento

**Corso:** MLOps e Monitoraggio con Python · **Blocco B**
**Tempo stimato:** 105 minuti · **Capitolo del manuale:** `manual/11_drift_reentrenamiento_IT.md`

## Cosa vedrai in questa lezione

- La differenza tra **data drift** (cambia la distribuzione delle variabili di ingresso) e **concept drift** (cambia la relazione tra le variabili e l'etichetta), con un piccolo esempio di ciascuno.
- Come misurare il *drift* con due test statistici semplici implementati da te: **Kolmogorov-Smirnov** (`scipy.stats.ks_2samp`) e l'**indice di stabilità della popolazione** (PSI, calcolato a mano con bucket fissi).
- Come confermare le stesse conclusioni con la libreria **Evidently**, su un cambiamento REALE tra `churn.csv` e `churn_v2.csv` (non una simulazione a parole).
- Come riaddestrare il modello sul dato con *drift* e misurare il suo impatto reale sul test originale -- incluso un impatto che la metrica globale non mostra, ma un'analisi per segmento sì.
- Come definire un **innesco di riaddestramento** esplicito, e perché riaddestrare con dati già contaminati può perpetuare il problema che ha causato il *drift*.

Tutto avviene dentro una cartella di lavoro effimera (`proyecto_churn/`), creata all'inizio ed **eliminata alla fine** di questo stesso notebook. Nessun numero di questa lezione viene misurato con l'orologio reale: tutto proviene da dati con seed fisso.

## Prerequisiti

- Dalle lezioni 4, 5, 7 e 10: la configurazione vincitrice del modello (regressione logistica, `class_weight="balanced"`, `C=1,0`) e lo stesso set di test di `churn.csv`.
- Dalla lezione 6: l'idea che una metrica aggregata possa nascondere un problema reale in un sottogruppo (qui si ripete con il segmento "plan premium").
- Dalla lezione 10: la distinzione tra metriche di sistema e di modello -- questa lezione riprende esattamente la domanda lasciata aperta: cosa fare quando la distribuzione delle previsioni CAMBIA davvero.
- Python di base e `pandas`. Non serve aver usato Evidently prima: questa lezione la introduce da zero.

> **Come usare questo notebook:** esegui le celle in ordine, dall'alto in basso, una sola volta. Richiede meno di un minuto in totale.

---
## Installazione

Installa solo ciò che manca (Evidently, scipy).

In [ ]:
import warnings

# Silenziamo avvisi di compatibilità fissi (non aggiungono nulla qui) perché
# l'output di questo notebook non dipenda da dettagli interni di versioni delle librerie
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import subprocess
import sys


def asegurar(paquete, modulo):
    """Installa il pacchetto solo se manca (idempotente e silenzioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("scikit-learn", "sklearn")
asegurar("scipy", "scipy")
asegurar("matplotlib", "matplotlib")
asegurar("evidently", "evidently")
print("Librerie pronte.")

---
## Cella 0 — Setup

In [ ]:
import os
import shutil
import json
import warnings
import logging

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import joblib
from scipy.stats import ks_2samp
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score

# Silenziamo avvisi fissi di compatibilità (non aggiungono nulla qui) perché
# l'output di questo notebook non dipenda da dettagli interni di versioni delle librerie
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger("evidently").setLevel(logging.ERROR)

def cargar_datos(nombre):
    """Cerca un file di dati in datos/, ../datos/ e /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/18-mlops-monitoraggio/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "Non trovo " + nombre + ". Carica la cartella datos/ del corso (in Colab: pannello dei file) o lasciala accanto al notebook.")

SEMILLA = 42
COLUMNAS_CATEGORICAS = ["plan", "contrato", "metodo_pago"]
COLUMNAS_NUMERICAS = ["antiguedad_meses", "uso_mensual_horas", "tickets_soporte", "factura_mensual"]
COLUMNAS_ENTRADA = COLUMNAS_CATEGORICAS + COLUMNAS_NUMERICAS

print("Librerie e funzioni ausiliarie pronte.")

---
## Cartella di lavoro effimera (si pulisce all'inizio e alla fine)

In [ ]:
CARPETA = "proyecto_churn"

# Puliamo e ricreiamo la cartella perché il notebook sia ripetibile da zero (anche all'inizio, nel caso sia rimasto qualcosa da un'esecuzione precedente)
if os.path.exists(CARPETA):
    shutil.rmtree(CARPETA)
os.makedirs(CARPETA)
print("Cartella di lavoro creata:", os.path.abspath(CARPETA))

---
## Caricare le due versioni dei dati (sola lettura)

In [ ]:
RUTA_V1 = cargar_datos("churn.csv")
RUTA_V2 = cargar_datos("churn_v2.csv")
df_v1 = pd.read_csv(RUTA_V1)
df_v2 = pd.read_csv(RUTA_V2)

print("churn.csv (riferimento):", df_v1.shape)
print("churn_v2.csv (nuovo):", df_v2.shape)
assert list(df_v1.columns) == list(df_v2.columns), "le due versioni devono avere lo stesso schema di colonne"
print("Stesso schema di colonne in entrambi i file:", list(df_v1.columns))

---
## 1. Data drift contro concept drift

- **Data drift**: cambia P(X), la distribuzione delle variabili di ingresso. La relazione P(Y|X) -- come quelle variabili predicono l'etichetta -- resta la stessa.
- **Concept drift**: cambia P(Y|X). La relazione tra le variabili e l'etichetta cambia, anche se le variabili di ingresso hanno esattamente la stessa distribuzione.

> **Nota:** entrambi i tipi di *drift* possono verificarsi insieme, e in pratica è spesso difficile separarli senza l'etichetta reale. I due esempi seguenti sono deliberatamente piccoli perché la differenza si veda senza ambiguità.

In [ ]:
# Esempio A -- DATA DRIFT: la distribuzione di ingresso cambia, la relazione con l'etichetta NO.
# Due popolazioni piccole di 4 clienti, con la STESSA regola di etichettatura: abbandona se tickets_soporte >= 3.
poblacion_antes = pd.DataFrame({"tickets_soporte": [0, 1, 1, 2], "abandono": [0, 0, 0, 0]})
poblacion_despues = pd.DataFrame({"tickets_soporte": [3, 4, 3, 5], "abandono": [1, 1, 1, 1]})

regla_es_la_misma = True  # "abbandona se tickets_soporte >= 3" non è cambiata: è cambiato solo CHI compare nei dati
for df_pob in (poblacion_antes, poblacion_despues):
    etiqueta_prevista = (df_pob["tickets_soporte"] >= 3).astype(int)
    regla_es_la_misma &= etiqueta_prevista.equals(df_pob["abandono"])

print("Media di tickets_soporte prima:", poblacion_antes["tickets_soporte"].mean(),
      "-- dopo:", poblacion_despues["tickets_soporte"].mean())
print("La regola 'tickets_soporte >= 3 => abandono' spiega ancora l'etichetta in entrambe le popolazioni?", regla_es_la_misma)
print("Questo è DATA DRIFT: è cambiata P(X) (la media di tickets_soporte è passata da 1,0 a 3,75), non P(Y|X).")

print()

# Esempio B -- CONCEPT DRIFT: la distribuzione di ingresso NON cambia, la relazione con l'etichetta SÌ.
# Immaginiamo che l'azienda introduca un rimborso automatico dopo il terzo ticket: la STESSA quantità di ticket non predice più l'abbandono allo stesso modo.
poblacion_antes_politica = pd.DataFrame({"tickets_soporte": [3, 4, 3, 4], "abandono": [1, 1, 1, 1]})
poblacion_despues_politica = pd.DataFrame({"tickets_soporte": [3, 4, 3, 4], "abandono": [0, 0, 0, 1]})

misma_distribucion_x = poblacion_antes_politica["tickets_soporte"].tolist() == poblacion_despues_politica["tickets_soporte"].tolist()
tasa_abandono_antes = poblacion_antes_politica["abandono"].mean()
tasa_abandono_despues = poblacion_despues_politica["abandono"].mean()

print("I valori di tickets_soporte sono identici prima e dopo?", misma_distribucion_x)
print("Tasso di abbandono tra clienti con >=3 ticket, prima:", tasa_abandono_antes,
      "-- dopo la nuova politica di rimborso:", tasa_abandono_despues)
print("Questo è CONCEPT DRIFT: P(X) non è cambiata (stessi tickets_soporte), ma P(Y|X) sì (dal 100% al 25% di abbandono con >=3 ticket).")

assert misma_distribucion_x and tasa_abandono_antes == 1.0 and tasa_abandono_despues == 0.25

**Cosa osservi?** Nell'esempio A la media di `tickets_soporte` passa da 1,0 a 3,75 -- P(X) è chiaramente cambiata -- ma la stessa regola ("3 o più ticket implica abbandono") continua a predire l'etichetta esatta nelle due popolazioni: è *data drift* puro. Nell'esempio B, i valori di `tickets_soporte` sono identici nelle due popolazioni -- P(X) non è cambiata affatto -- ma la frazione di clienti che abbandona con 3 o più ticket scende dal 100 % al 25 % dopo il cambio di politica: è *concept drift* puro. In un caso reale quasi mai sono così puliti né così separabili; il punto di questo esempio è solo fissare la differenza concettuale prima di misurarla con dati reali.

---
## 2. L'indice di stabilità della popolazione (PSI), a mano

Prima di applicarlo a dati reali, un esempio minimo verificabile a mano: 4 bucket, con una percentuale di riferimento e una percentuale nuova già calcolate.

In [ ]:
# Esempio minimo di PSI, verificabile a mano, con 4 bucket e percentuali già calcolate (non conteggi grezzi).
porcentaje_referencia = np.array([0.40, 0.30, 0.20, 0.10])
porcentaje_nuevo = np.array([0.10, 0.20, 0.30, 0.40])


def calcular_psi(porcentaje_referencia, porcentaje_nuevo):
    """PSI = somma, in ogni bucket, di (nuovo% - riferimento%) * ln(nuovo% / riferimento%)."""
    return float(np.sum((porcentaje_nuevo - porcentaje_referencia) * np.log(porcentaje_nuevo / porcentaje_referencia)))


psi_ejemplo = calcular_psi(porcentaje_referencia, porcentaje_nuevo)
print("PSI dell'esempio (4 bucket):", round(psi_ejemplo, 5))

# Verifica a mano, bucket per bucket: (nuovo-rif)*ln(nuovo/rif)
aportes_a_mano = [
    (0.10 - 0.40) * np.log(0.10 / 0.40),  # ≈0,41589
    (0.20 - 0.30) * np.log(0.20 / 0.30),  # ≈0,04055
    (0.30 - 0.20) * np.log(0.30 / 0.20),  # ≈0,04055
    (0.40 - 0.10) * np.log(0.40 / 0.10),  # ≈0,41589
]
print("Contributi per bucket (a mano):", [round(a, 5) for a in aportes_a_mano])
print("Somma dei contributi:", round(sum(aportes_a_mano), 5))
assert abs(sum(aportes_a_mano) - psi_ejemplo) < 1e-9, "la funzione e il calcolo a mano devono coincidere"
assert psi_ejemplo > 0.25, "questo esempio è stato progettato per cadere nella zona di drift severo (PSI>0,25)"
print("PSI > 0,25: con la convenzione abituale, questo spostamento si classificherebbe come drift SEVERO.")

---
## 3. PSI e Kolmogorov-Smirnov reali: `churn.csv` contro `churn_v2.csv`

Applichiamo `calcular_psi` (avvolto in `psi_numerico`/`psi_categorico`, che costruiscono i bucket) e `scipy.stats.ks_2samp` a 5 colonne: 3 che sono state spostate apposta generando `churn_v2.csv` (`uso_mensual_horas`, `tickets_soporte`, `contrato`) e 2 colonne di controllo che NON sono state toccate (`antiguedad_meses`, `factura_mensual`).

> **Attenzione:** i bucket del PSI si fissano sempre con i quantili del RIFERIMENTO (`churn.csv`), mai con quelli del dato nuovo -- se si ricalcolassero sul dato nuovo, il PSI potrebbe uscire artificialmente basso anche con uno spostamento reale.

In [ ]:
def psi_numerico(referencia, nuevo, buckets=10):
    """PSI per una colonna numerica: i bordi dei bucket si fissano con i quantili del RIFERIMENTO (mai del nuovo dato), e si riusano tali e quali per contare il nuovo."""
    bordes = np.unique(np.quantile(referencia, np.linspace(0, 1, buckets + 1)))
    bordes[0], bordes[-1] = -np.inf, np.inf
    cuenta_ref, _ = np.histogram(referencia, bins=bordes)
    cuenta_nuevo, _ = np.histogram(nuevo, bins=bordes)
    porc_ref = np.clip(cuenta_ref / len(referencia), 1e-4, None)
    porc_nuevo = np.clip(cuenta_nuevo / len(nuevo), 1e-4, None)
    return calcular_psi(porc_ref, porc_nuevo)


def psi_categorico(referencia, nuevo):
    """PSI per una colonna categorica: ogni categoria è un bucket."""
    categorias = sorted(set(referencia.unique()) | set(nuevo.unique()))
    porc_ref = np.array([np.clip((referencia == c).mean(), 1e-4, None) for c in categorias])
    porc_nuevo = np.array([np.clip((nuevo == c).mean(), 1e-4, None) for c in categorias])
    return calcular_psi(porc_ref, porc_nuevo)


COLUMNAS_A_COMPROBAR = ["uso_mensual_horas", "tickets_soporte", "contrato", "antiguedad_meses", "factura_mensual"]
DESPLAZADAS_A_PROPOSITO = {"uso_mensual_horas", "tickets_soporte", "contrato"}

resultados = []
for columna in COLUMNAS_A_COMPROBAR:
    if columna == "contrato":
        psi = psi_categorico(df_v1[columna], df_v2[columna])
        ks_stat, ks_p = None, None
    else:
        psi = psi_numerico(df_v1[columna].astype(float), df_v2[columna].astype(float))
        ks_res = ks_2samp(df_v1[columna], df_v2[columna])
        ks_stat, ks_p = float(ks_res.statistic), float(ks_res.pvalue)
    resultados.append({
        "columna": columna,
        "desplazada_a_proposito": columna in DESPLAZADAS_A_PROPOSITO,
        "psi": round(psi, 4),
        "ks_estadistico": None if ks_stat is None else round(ks_stat, 4),
        "ks_p_valor": ks_p,
    })

tabla_drift = pd.DataFrame(resultados)
pd.set_option("display.width", 140)
print(tabla_drift.to_string(index=False))

---
## 4. Interpretare il PSI con una convenzione standard

In [ ]:
def interpretar_psi(psi):
    """Convenzione standard (Siddiqi, "Credit Risk Scorecards"): <0,1 nessun drift; 0,1-0,25 drift moderato; >0,25 drift severo."""
    if psi < 0.10:
        return "nessun drift"
    if psi < 0.25:
        return "drift moderato"
    return "drift severo"


tabla_drift["interpretacion"] = tabla_drift["psi"].apply(interpretar_psi)
print(tabla_drift[["columna", "desplazada_a_proposito", "psi", "interpretacion"]].to_string(index=False))

for _, fila in tabla_drift.iterrows():
    if fila["desplazada_a_proposito"]:
        assert fila["psi"] > 0.10, "una colonna spostata apposta dovrebbe mostrare almeno drift moderato"
    else:
        assert fila["psi"] < 0.10, "una colonna di controllo non dovrebbe mostrare drift"
print()
print("Le 3 colonne spostate apposta mostrano drift moderato o severo; le 2 colonne di controllo (non toccate generando churn_v2.csv) non mostrano nessun drift. PSI e KS distinguono correttamente lo spostamento reale dal rumore di campionamento.")

---
## 5. La stessa conclusione con Evidently

`Report([DataDriftPreset()])` confronta automaticamente ogni colonna con il metodo adatto al suo tipo (distanza di Wasserstein normalizzata per le numeriche, distanza di Jensen-Shannon per le categoriche) e una soglia di default di 0,10.

In [ ]:
from evidently import Report, Dataset, DataDefinition
from evidently.presets import DataDriftPreset

conjunto_referencia = Dataset.from_pandas(df_v1[COLUMNAS_ENTRADA], data_definition=DataDefinition())
conjunto_nuevo = Dataset.from_pandas(df_v2[COLUMNAS_ENTRADA], data_definition=DataDefinition())

informe = Report([DataDriftPreset()])
resultado = informe.run(reference_data=conjunto_referencia, current_data=conjunto_nuevo)
resultado_dict = resultado.dict()

RUTA_INFORME = os.path.join(CARPETA, "informe_drift_evidently.json")
with open(RUTA_INFORME, "w", encoding="utf-8") as f:
    json.dump({"metrics": [{"nombre": m["metric_name"], "valor": (m["value"] if isinstance(m["value"], dict) else float(m["value"]))} for m in resultado_dict["metrics"]]}, f, ensure_ascii=False, indent=2, default=float)

filas_evidently = []
for m in resultado_dict["metrics"]:
    if m["metric_name"].startswith("DriftedColumnsCount"):
        continue
    columna = m["config"]["column"]
    valor = float(m["value"])
    filas_evidently.append({"columna": columna, "metodo": m["config"]["method"], "distancia": round(valor, 4),
                             "supera_umbral_0_10": valor > 0.10})

tabla_evidently = pd.DataFrame(filas_evidently).sort_values("columna").reset_index(drop=True)
print(tabla_evidently.to_string(index=False))

conteo_drift = next(m["value"] for m in resultado_dict["metrics"] if m["metric_name"].startswith("DriftedColumnsCount"))
print()
print("Colonne segnalate con drift da Evidently:", int(conteo_drift["count"]), "di 7 colonne di ingresso (", round(conteo_drift["share"] * 100, 1), "%).")

columnas_marcadas = set(tabla_evidently.loc[tabla_evidently["supera_umbral_0_10"], "columna"])
assert columnas_marcadas == DESPLAZADAS_A_PROPOSITO, "Evidently dovrebbe segnalare esattamente le 3 colonne spostate apposta"
print("Evidently, con il proprio metodo per colonna (Wasserstein per le numeriche, Jensen-Shannon per le categoriche), segnala ESATTAMENTE le stesse 3 colonne del nostro PSI/KS manuale -- due modi di misurare indipendenti, stessa conclusione.")

---
## 6. Riaddestrare sul dato con *drift* e misurare l'impatto reale

Riaddestriamo la stessa Pipeline (regressione logistica, `class_weight="balanced"`, `C=1,0`) ma ora su `churn_v2.csv`, e la valutiamo sullo STESSO set di test di `churn.csv` che usano le lezioni 4, 5, 7 e 10 -- così il confronto è limpio: stessi dati di test, cambia solo con cosa è stato addestrato.

In [ ]:
def construir_pipeline():
    preprocesador = ColumnTransformer([
        ("categoricas", OneHotEncoder(handle_unknown="ignore"), COLUMNAS_CATEGORICAS),
        ("numericas", StandardScaler(), COLUMNAS_NUMERICAS),
    ])
    clasificador = LogisticRegression(class_weight="balanced", random_state=SEMILLA, max_iter=1000, C=1.0)
    return Pipeline([("preprocesado", preprocesador), ("clasificador", clasificador)])


# Il set di test di churn.csv è lo stesso delle lezioni 4, 5, 7 e 10 (stesso seed, stesso test_size): è il punto di confronto fisso di tutto questo capitolo.
X_v1 = df_v1[COLUMNAS_ENTRADA]
y_v1 = df_v1["abandono"]
X_train, X_test, y_train, y_test = train_test_split(X_v1, y_v1, test_size=0.25, random_state=SEMILLA, stratify=y_v1)

modelo_original = construir_pipeline()
modelo_original.fit(X_train, y_train)
f1_original = f1_score(y_test, modelo_original.predict(X_test))
auc_original = roc_auc_score(y_test, modelo_original.predict_proba(X_test)[:, 1])

modelo_reentrenado = construir_pipeline()
modelo_reentrenado.fit(df_v2[COLUMNAS_ENTRADA], df_v2["abandono"])
f1_reentrenado = f1_score(y_test, modelo_reentrenado.predict(X_test))
auc_reentrenado = roc_auc_score(y_test, modelo_reentrenado.predict_proba(X_test)[:, 1])

joblib.dump(modelo_reentrenado, os.path.join(CARPETA, "modelo_reentrenado.pkl"))

print("Modello addestrato su churn.csv, valutato sul TEST di churn.csv (linea base, lezioni 4/5/7/10):")
print("  F1:", round(f1_original, 4), " AUC:", round(auc_original, 4))
print("Modello RIADDESTRATO su churn_v2.csv (col drift), valutato sullo STESSO test di churn.csv:")
print("  F1:", round(f1_reentrenado, 4), " AUC:", round(auc_reentrenado, 4))
print("Differenza di F1:", round(f1_reentrenado - f1_original, 4))

---
## 7. La stessa domanda della lezione 6: cosa nasconde la media?

In [ ]:
# La differenza globale di F1 è piccola -- esattamente lo stesso rischio della lezione 6: una metrica aggregata può nascondere un problema reale in un sottogruppo. Ripetiamo l'analisi per segmento di "plan".
X_test_con_etiqueta = X_test.copy()
X_test_con_etiqueta["abandono_real"] = y_test.values

filas_segmento = []
for segmento in sorted(df_v1["plan"].unique()):
    mascara = X_test_con_etiqueta["plan"] == segmento
    y_segmento = X_test_con_etiqueta.loc[mascara, "abandono_real"]
    X_segmento = X_test_con_etiqueta.loc[mascara, COLUMNAS_ENTRADA]
    f1_orig_seg = f1_score(y_segmento, modelo_original.predict(X_segmento))
    f1_reen_seg = f1_score(y_segmento, modelo_reentrenado.predict(X_segmento))
    filas_segmento.append({"plan": segmento, "n_clientes": int(mascara.sum()),
                            "F1_modelo_original": round(f1_orig_seg, 4),
                            "F1_modelo_reentrenado": round(f1_reen_seg, 4)})

tabla_segmentos = pd.DataFrame(filas_segmento)
print(tabla_segmentos.to_string(index=False))

peor_caso = tabla_segmentos.loc[tabla_segmentos["F1_modelo_reentrenado"].idxmin()]
assert peor_caso["plan"] == "premium" and peor_caso["F1_modelo_reentrenado"] < peor_caso["F1_modelo_original"], "il segmento premium dovrebbe peggiorare chiaramente dopo il riaddestramento con il dato spostato"
print()
print("Il segmento", peor_caso["plan"], "passa da F1", peor_caso["F1_modelo_original"],
      "a F1", peor_caso["F1_modelo_reentrenado"],
      "dopo il riaddestramento: il modello riaddestrato smette di rilevare QUALSIASI abbandono reale in quel segmento, anche se l'F1 globale cambia appena. È un impatto reale, non solo una differenza statistica nelle colonne di ingresso.")

---
## 8. Innesco di riaddestramento

Un innesco esplicito, con due condizioni indipendenti: `PSI > 0,25` in almeno una colonna, O l'F1 misurato scende sotto una soglia minima.

In [ ]:
def evaluar_disparador(tabla_drift, f1_actual, umbral_psi=0.25, umbral_f1=0.25):
    """Innesco esplicito: riaddestrare se (a) almeno una colonna chiave supera PSI>umbral_psi, O (b) l'F1 misurato scende sotto umbral_f1. Restituisce quale condizione si è verificata."""
    columnas_con_drift_severo = tabla_drift.loc[tabla_drift["psi"] > umbral_psi, "columna"].tolist()
    disparado_por_drift = len(columnas_con_drift_severo) > 0
    disparado_por_calidad = f1_actual < umbral_f1
    return {
        "disparar_reentrenamiento": disparado_por_drift or disparado_por_calidad,
        "disparado_por_drift": disparado_por_drift,
        "columnas_con_drift_severo": columnas_con_drift_severo,
        "disparado_por_calidad": disparado_por_calidad,
    }


decision = evaluar_disparador(tabla_drift, f1_original)
print("Valutazione dell'innesco (con le cifre di questo stesso notebook):")
for clave, valor in decision.items():
    print(" ", clave, ":", valor)

assert decision["disparar_reentrenamiento"] and decision["disparado_por_drift"] and not decision["disparado_por_calidad"], \
    "in questo esempio l'innesco dovrebbe attivarsi per drift, non per caduta dell'F1 globale"
print()
print("L'innesco si attiva per DRIFT (PSI>0,25 su", decision["columnas_con_drift_severo"], "), NON per caduta dell'F1 globale -- che infatti non è caduto. Se sorvegliassimo solo l'F1 aggregato, come avvertiva la lezione 10, questo problema passerebbe inosservato finché un intero sottogruppo (plan premium) smettesse di essere rilevato.")

---
## 9. Il rischio di riaddestrare con dati contaminati

Questo capitolo ha riaddestrato assumendo che `churn_v2.csv` rifletta un cambiamento reale e legittimo nel comportamento dei clienti (più contratti mensili, meno utilizzo, più ticket). Ma uno spostamento con questa stessa forma statistica potrebbe anche venire da un **bug di raccolta dati** -- per esempio, un guasto nel sistema di fatturazione che sottoconta `uso_mensual_horas` per i contratti mensili, o un cambiamento non annunciato in come si classificano i ticket. PSI e KS non distinguono queste due cause: dicono solo che P(X) è cambiata, non PERCHÉ.

> **Attenzione:** riaddestrare automaticamente non appena scatta la soglia, senza indagare la causa, rischia di perpetuare il problema: se il drift viene da un bug, il modello riaddestrato apprende una relazione contaminata da quel bug, e ogni riaddestramento successivo sullo stesso flusso di dati difettoso lo riafferma invece di correggerlo. L'innesco della sezione 8 dice QUANDO indagare; non sostituisce l'indagine -- solo un ingegnere dei dati o chi gestisce la fonte può confermare se il cambiamento è reale o un difetto.

---
## 10. Chiudendo il ciclo del corso

Se il team conferma che il drift è reale (non un bug) e decide di riaddestrare, il modello riaddestrato non si distribuisce direttamente: torna a passare per lo stesso ciclo che abbiamo già costruito in questo corso -- **versionamento** (lezione 3: una nuova versione dei dati e del modello, con il proprio hash), **tracciamento** (lezione 4: un nuovo esperimento registrato in MLflow, confrontabile con i precedenti), **test** (lezione 6: la stessa suite di `pytest` e `pandera` deve tornare a passare, incluso il controllo dell'F1 minimo) e **distribuzione** (lezione 9: canary o shadow prima di sostituire il modello in produzione, mai una sostituzione diretta). Il *drift* non è un evento isolato: è il segnale che riavvia lo stesso ciclo, non una scorciatoia per saltarlo.

---
## 11. Riepilogo e chiusura

- Abbiamo distinto *data drift* (cambia P(X)) da *concept drift* (cambia P(Y|X)) con un piccolo esempio di ciascuno.
- Abbiamo implementato il PSI a mano (con un esempio verificabile) e usato il KS di `scipy`, su un cambiamento reale tra `churn.csv` e `churn_v2.csv`: le 3 colonne spostate apposta mostrano drift, le 2 di controllo no.
- Abbiamo confermato la stessa conclusione con Evidently, con metodi diversi per tipo di colonna.
- Abbiamo riaddestrato sul dato con drift e misurato un impatto reale, non solo statistico: l'F1 globale cambia appena, ma un intero segmento (plan premium) smette di essere rilevato.
- Abbiamo definito un innesco esplicito (PSI o F1) e discusso il rischio di riaddestrare su dati già contaminati dal problema che ha causato il drift.
- Abbiamo chiuso il ciclo: il modello riaddestrato torna a passare per versionamento (lezione 3), tracciamento (lezione 4), test (lezione 6) e distribuzione (lezione 9).

*Continua con la lezione 12: Gobernanza, costi e LLMOps + progetto.*

---
## Pulizia finale

In [ ]:
shutil.rmtree(CARPETA)
print("Cartella di lavoro eliminata:", CARPETA)
assert not os.path.exists(CARPETA)